# 20.12 — Knowledge distillation

Distillation trains a smaller student to match a larger teacher, preserving useful probability structure while meeting production budgets.

Softmax and cross-entropy supply the training signal; compression and serving budgets supply the deployment motivation. This notebook computes the actual temperature probabilities, blended distillation loss, calibration, routing, and speedup trade-offs.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2017)


def percentile(values, q):
    return float(np.percentile(np.asarray(values, dtype=float), q))


def softmax(logits, temperature=1.0):
    scaled = np.asarray(logits, dtype=float) / temperature
    shifted = scaled - np.max(scaled, axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values, axis=-1, keepdims=True)


def cross_entropy(target_probs, pred_probs):
    clipped = np.clip(pred_probs, 1e-9, 1.0)
    return float(-np.sum(target_probs * np.log(clipped)))


def expected_calibration_error(confidence, correct, bins=10):
    confidence = np.asarray(confidence, dtype=float)
    correct = np.asarray(correct, dtype=float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    total = len(confidence)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidence >= lo) & (confidence < hi)
        if hi == 1.0:
            mask = (confidence >= lo) & (confidence <= hi)
        if np.any(mask):
            gap = abs(float(np.mean(confidence[mask])) - float(np.mean(correct[mask])))
            ece += float(np.mean(mask)) * gap
    return ece


def make_f17_workload_ladder(seed=2017):
    local_rng = np.random.default_rng(seed)
    return [
        {
            "rung": "D1",
            "name": "tiny hand-check",
            "requests": 60,
            "precision": "int8",
            "scale": 1.8 / 127.0,
            "shape": (8,),
            "load": 1.0,
            "data": np.array([0.73, -0.40, 1.20, -1.70, 0.05, 0.88, -0.91, 1.55]),
        },
        {
            "rung": "D2",
            "name": "clean tensor workload",
            "requests": 400,
            "precision": "int8",
            "scale": 2.5 / 127.0,
            "shape": (64, 32),
            "load": 1.7,
            "data": local_rng.normal(0.0, 0.65, size=(64, 32)),
        },
        {
            "rung": "D3",
            "name": "outliers and bad calibration",
            "requests": 900,
            "precision": "int8",
            "scale": 1.0 / 127.0,
            "shape": (128, 48),
            "load": 2.4,
            "data": np.vstack([
                local_rng.normal(0.0, 0.45, size=(124, 48)),
                local_rng.normal(0.0, 2.2, size=(4, 48)),
            ]),
        },
        {
            "rung": "D4",
            "name": "digits-like classifier weights",
            "requests": 1600,
            "precision": "mixed int8/fp16",
            "scale": 3.0 / 127.0,
            "shape": (10, 64),
            "load": 3.5,
            "data": local_rng.normal(0.0, 0.85, size=(10, 64)),
        },
        {
            "rung": "D5",
            "name": "production-scale load simulation",
            "requests": 5000,
            "precision": "per-channel int8",
            "scale": 4.0 / 127.0,
            "shape": (256, 128),
            "load": 6.0,
            "data": local_rng.normal(0.0, 0.95, size=(256, 128)),
        },
    ]


def preview_ladder(ladder):
    for rung in ladder:
        data = rung.get("data")
        shape = getattr(data, "shape", rung.get("shape"))
        print(f"{rung['rung']} | {rung['name']} | shape={shape} | requests={rung['requests']} | precision={rung['precision']}")
        print("sample:", np.asarray(data).reshape(-1)[:5])


## The concept, built once (D1)

The lesson formula is $L=\alpha L_{soft}(p_T^T,p_S^T)+(1-\alpha)L_{hard}(y,p_S)$. Temperature softens teacher probabilities, so the class-1 soft target is $2.718/5.367=0.506$.

In [ ]:

def distill_loss(soft_loss, hard_loss, alpha, temperature=1.0, scale_temperature=False):
    scaled_soft_loss = soft_loss * temperature * temperature if scale_temperature else soft_loss
    return alpha * scaled_soft_loss + (1.0 - alpha) * hard_loss

teacher_exp = np.array([math.e ** 1.0, math.e ** 0.5, math.e ** 0.0])
teacher_probs = teacher_exp / np.sum(teacher_exp)
soft_target_class_1 = float(teacher_probs[0])
blend = distill_loss(0.42, 0.65, 0.7)
size_ratio = 12 / 120
speedup = 90 / 18

print("soft target", round(soft_target_class_1, 3))
print("blend", round(blend, 3))
print("size ratio", round(size_ratio, 2))
print("speedup", round(speedup, 1))

assert round(float(np.sum(teacher_exp)), 3) == 5.367
assert round(soft_target_class_1, 3) == 0.506
assert round(blend, 3) == 0.489
assert round(size_ratio, 2) == 0.10
assert round(speedup, 1) == 5.0


Now reuse the loss inside a production evaluator. It compares top-1 agreement, calibration, student compression, and p95 latency for each rung.

In [ ]:

def make_distillation_ladder(seed=2017):
    local_rng = np.random.default_rng(seed + 12)
    ladder = make_f17_workload_ladder(seed + 12)
    for i, rung in enumerate(ladder):
        n = rung["requests"]
        classes = 3 + i
        teacher_logits = local_rng.normal(0.0, 1.4, size=(n, classes))
        noise = 0.15 + 0.16 * i
        student_logits = teacher_logits * (0.96 - 0.07 * i) + local_rng.normal(0.0, noise, size=(n, classes))
        labels = np.argmax(teacher_logits + local_rng.normal(0.0, 0.2, size=(n, classes)), axis=1)
        rung["teacher_logits"] = teacher_logits
        rung["student_logits"] = student_logits
        rung["labels"] = labels
        rung["teacher_latency_ms"] = 90.0 + 12.0 * i
        rung["student_latency_ms"] = 18.0 + 5.0 * i
        rung["teacher_params_m"] = 120.0 + 20.0 * i
        rung["student_params_m"] = 12.0 + 5.0 * i
    return ladder


def evaluate_distillation_workload(rung, alpha=0.7, temperature=2.0):
    teacher_probs = softmax(rung["teacher_logits"], temperature)
    student_probs = softmax(rung["student_logits"], temperature)
    hard_probs = softmax(rung["student_logits"], 1.0)
    teacher_top = np.argmax(teacher_probs, axis=1)
    student_top = np.argmax(student_probs, axis=1)
    labels = rung["labels"]
    soft_losses = -np.sum(teacher_probs * np.log(np.clip(student_probs, 1e-9, 1.0)), axis=1)
    hard_losses = -np.log(np.clip(hard_probs[np.arange(len(labels)), labels], 1e-9, 1.0))
    blended = alpha * soft_losses * temperature * temperature + (1.0 - alpha) * hard_losses
    confidence = np.max(hard_probs, axis=1)
    correct = student_top == labels
    agreement = float(np.mean(student_top == teacher_top))
    accuracy = float(np.mean(correct))
    ece = expected_calibration_error(confidence, correct)
    compression = rung["teacher_params_m"] / rung["student_params_m"]
    latency_samples = rng.normal(rung["student_latency_ms"], 2.5 + rung["load"], size=300)
    p95_latency_ms = percentile(latency_samples, 95)
    return {
        "loss": float(np.mean(blended)),
        "agreement": agreement,
        "accuracy": accuracy,
        "ece": ece,
        "compression": compression,
        "speedup": rung["teacher_latency_ms"] / rung["student_latency_ms"],
        "p95_latency_ms": p95_latency_ms,
        "artifact": confidence[:80],
    }


## The dataset ladder

D1 uses three teacher classes, D2 is cleaner synthetic classification, D3 stresses temperature and calibration, D4 mimics a small digits classifier, and D5 simulates production teacher/student routing.

In [ ]:

ladder = make_distillation_ladder()
preview_ladder(ladder)


## Run the SAME method across D1–D5

In [ ]:

distill_results = []
for rung in ladder:
    result = evaluate_distillation_workload(rung)
    result["rung"] = rung["rung"]
    result["name"] = rung["name"]
    distill_results.append(result)

print("rung | loss | accuracy | agreement | ece | compression | speedup | p95_ms")
for result in distill_results:
    print(f"{result['rung']} | {result['loss']:.3f} | {result['accuracy']:.3f} | {result['agreement']:.3f} | {result['ece']:.3f} | {result['compression']:.1f}x | {result['speedup']:.1f}x | {result['p95_latency_ms']:.1f}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, result in zip(axes[0], distill_results):
    ax.plot(result["artifact"], linewidth=1.0)
    ax.set_title(result["rung"] + " confidence")
    ax.set_ylim(0.0, 1.0)

metric_curve = [result["compression"] for result in distill_results]
axes[1, 0].plot([rung["rung"] for rung in ladder], metric_curve, marker="o")
axes[1, 0].set_title("compression curve")
for ax, result in zip(axes[1, 1:], distill_results[1:]):
    ax.bar(["p95", "speedup", "ece"], [result["p95_latency_ms"], result["speedup"], result["ece"]])
    ax.set_title(result["rung"] + " operational panel")
plt.tight_layout()


## Pitfall on D5: only measuring accuracy

A student can match top-1 while becoming overconfident. Reproduce an accuracy-only pass, then add calibration as a launch gate.

In [ ]:

d5 = ladder[-1]
d5_good = evaluate_distillation_workload(d5)
miscalibrated = dict(d5)
miscalibrated["student_logits"] = d5["student_logits"] * 2.8
bad = evaluate_distillation_workload(miscalibrated)
accuracy_gate = abs(bad["accuracy"] - d5_good["accuracy"]) <= 0.02
calibration_gate = bad["ece"] <= d5_good["ece"] + 0.02
fixed_launch_decision = accuracy_gate and calibration_gate

print("good accuracy/ece", round(d5_good["accuracy"], 3), round(d5_good["ece"], 3))
print("bad accuracy/ece", round(bad["accuracy"], 3), round(bad["ece"], 3))
print("accuracy-only would pass", accuracy_gate)
print("calibration-gated launch", fixed_launch_decision)
assert accuracy_gate
assert bad["ece"] > d5_good["ece"]
assert not fixed_launch_decision


## Evaluate it + Practice

- Metric: track compression and latency speedup with calibration; compare with the no-skill baseline `teacher-only serving at 90 ms`.
- Cheap sanity check: temperature-2 teacher probabilities should sum to 1.
- Ablation: remove temperature scaling or calibration gate.
- Failure signals: ECE rises while top-1 remains flat.
- Production guardrail: monitor the metric by rung instead of averaging D1 with D5.

Practice 1: Change one D5 load or precision setting and predict the metric before running.

Practice 2: Add one operational constraint, such as memory budget or p95 latency budget, and mark the first rung that violates it.

Practice 3: Write a one-line launch rule that uses the metric and one pitfall guardrail.